## This notebook analyzes biotic data collected throughout the study
Data from collected anemone tentacles were processed in the lab, quantifying algal density and chlorophyll concentration. Read the  `methodology` section in the publication for thorough data collection and processing details.

Duration of data collection: 7-11-2022 to 3-17-2023, though field collection and lab methodologies were still in development until the end of August.  
Data included in the analysis: 8-27-2022 to 3-17-2023  

## The notebook is broken into three steps:
### **Step 1) Pull and Clean Biotic Data**  
**Algal Density** - The number of algal cells per microgram of animal (anemone) protein  
**Chlorophyll Concentration** - How much chlorophyll was measured per microgram of animal (anemone) protein

### **Step 2) Determine Normalcy Status of the Variables we are Measuring** 
Dependent Variables: Algal Density & Chlorophyll Concentration
* Histogram
* Shapiro-Wilk test of normalcy

### **Step 3) Generate Figures & Statistics**
* Bar Plots
* Box Plots
* Kruskal-Wallis Test(s) - Non-parametric testing

In [ ]:
# load some library
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd
import scikit_posthocs as sp

sys.path.append(os.path.expanduser('../../'))
from utils.functions import normalcy, kruskal_drops
from utils.visuals import batch_bar_overlay, intertidal_box_plot, batch_box_plot

## **Step 1) Pull and Clean Biotic Data**

### Call csv file
* save master sheet as the current date in 'mmddyyy.csv' format
* make a new column called 'date' and takes my'date of collection' column and makes it a datetime format

In [ ]:
start_date = '2022-07-01T12:00:00'
end_date = '2023-04-06T12:00:00'

data_path = 'C:/Users/jespi/sfsu-masters/sfsu-masters-anthopleura-sola-1/biotic_data/simplified_master_data.csv'
a_sola_data = pd.read_csv(data_path)
a_sola_data['date_time'] = pd.to_datetime(a_sola_data['date_of_collection'])
a_sola_data = a_sola_data[(a_sola_data['date_time'] > start_date) & (a_sola_data['date_time'] <= end_date)]
a_sola_data['doy'] = a_sola_data['date_time'].dt.strftime('%j')
a_sola_data['doy'] = a_sola_data['doy'].astype(str).astype(int)

### Alga Density
Aside from collection groups 1-3 being removed from the analysis, three extreme outliers within the algal density metric were also subset out. 

In [ ]:
# Filter for non-null 'collection_group'
a_sola_algae_data = a_sola_data[a_sola_data['collection_group'].notnull()]

# Exclude specific 'collection_group' values and filter on 'num_cells_per_ug_protein'
a_sola_algae_data = a_sola_algae_data[
    (~a_sola_algae_data['collection_group'].isin([1.0, 2.0, 3.0])) &
    (a_sola_algae_data['num_cells_per_ug_protein'].notnull()) &
    (a_sola_algae_data['num_cells_per_ug_protein'] < 3000)
]
a_sola_algae_data = a_sola_algae_data.drop(columns='ng_chlorophyll_per_ug_protein')

# Extract 'doy' column and count the rows
algae_doy = a_sola_algae_data['doy']
num_rows = len(a_sola_algae_data)

print(num_rows)

### Chlorophyll
Aside from collection groups 1-3 being removed from the analysis, ......

Note - Prepared chlorophyll concentration samples from collection #7 (collection dates Oct. 11/12 2022) were contaminated and are missing from the dataset.

In [ ]:
# Filter for non-null 'collection_group'
a_sola_chlorophyll_data = a_sola_data[a_sola_data['collection_group'].notnull()]

# Exclude specific 'collection_group' values and filter on required columns
a_sola_chlorophyll_data = a_sola_chlorophyll_data[
    (~a_sola_chlorophyll_data['collection_group'].isin([1.0, 2.0, 3.0])) &
    (a_sola_chlorophyll_data['num_cells_per_ug_protein'].notnull()) &
    (a_sola_chlorophyll_data['ng_chlorophyll_per_ug_protein'].notnull()) &
    (a_sola_chlorophyll_data['num_cells_per_ug_protein'] < 3000)
]

# Extract 'doy' column and count the rows
chl_doy = a_sola_chlorophyll_data['doy']
num_rows = len(a_sola_chlorophyll_data)

print(num_rows)

## **Step 2) Determine Normalcy Status**

### Visualize each dataset shape via histogram

In [ ]:
fig, (ax1, ax2)  = plt.subplots(2, 1, sharey=False, figsize=(13,11))

ax1.hist(a_sola_algae_data.num_cells_per_ug_protein)
ax1.set(title ='Cells per Animal Protein')

ax2.hist(a_sola_chlorophyll_data.ng_chlorophyll_per_ug_protein)
ax2.set(title ='ng Chlorophyll per Animal Protein')

### Shapiro-Wilk test 
Check if data is normally distributed

p-value interpretation: 
* p > 0.05 means we fail to reject the null hypothesis, suggesting data is likely normal
* p ≤ 0.05 means we reject the null, indicating data significantly deviates from normality, making it non-normal.

Our interpretation of algal density and chlorophyll concentration datasets:
Both datasets p-values are significantly below the 0.05 threshold and their skewed histograms suggest both dependent datasets are **non-normal**

In [ ]:
print('Testing normalcy of algal density data:')
normalcy(a_sola_algae_data, 'num_cells_per_ug_protein')
print('')
print('Testing normalcy of chlorophyll concentration data:')
normalcy(a_sola_chlorophyll_data, 'ng_chlorophyll_per_ug_protein')

## **Step 3) Generate Figures & Statistics**

### Bar graphs
* group data by collection group and intertidal zone
* plot mean, SEM error bars
* indicate sample size for each intertidal zone

### Algal Density

In [ ]:
batch_bar_overlay(a_sola_algae_data, 'num_cells_per_ug_protein', save_path='figures/pre_annotated_figures/algal_tidal_bars_no_title.png', colors=['#a8ddb5', '#4eb3d3', '#08589e'])

### Chlorophyll Concentration

In [ ]:
batch_bar_overlay(a_sola_chlorophyll_data, 'ng_chlorophyll_per_ug_protein', save_path='figures/pre_annotated_figures/chlorophyll_tidal_bars_no_title.png', colors=['#c7e9b4', '#41b6c4', '#253494'])

### Box Plots
Generate box plot figures summarizing data by collection date across all tidal zones over time

Algal Density

In [ ]:
batch_box_plot(a_sola_algae_data, 'num_cells_per_ug_protein', 'Algal Cells/ug Animal Protein', 
               zone='', save_path='plots/algae_box_no_title', 
               box_colors='#4eb3d3') 

Chlorophyll Concentration

In [ ]:
batch_box_plot(a_sola_chlorophyll_data, 'ng_chlorophyll_per_ug_protein', 'ng Chl α/ug Animal Protein', 
               zone='', save_path='plots/chl_box_no_title',
               box_colors='#2c7fb8') 

### Kruskal-Wallis Test 
Pre vs post drop observed in the bar graph
* can use individual tidal heights or all

In [ ]:
# algal density
kruskal_drops(a_sola_algae_data, 'num_cells_per_ug_protein', 'all')

In [ ]:
# chlorophyll concentration
kruskal_drops(a_sola_chlorophyll_data, 'ng_chlorophyll_per_ug_protein', 'all')

### Testing statistical significance between intertidal zones

In [ ]:
data = [a_sola_algae_data[a_sola_algae_data['intertidal_zone']=="low"]['num_cells_per_ug_protein'],
        a_sola_algae_data[a_sola_algae_data['intertidal_zone']=="middle"]['num_cells_per_ug_protein'],
        a_sola_algae_data[a_sola_algae_data['intertidal_zone']=="high"]['num_cells_per_ug_protein']]

# using the posthoc_dunn() function
p_values= sp.posthoc_dunn(data, p_adjust = 'holm')

print(p_values)

In [ ]:
data = [a_sola_chlorophyll_data[a_sola_chlorophyll_data['intertidal_zone']=="low"]['ng_chlorophyll_per_ug_protein'],
        a_sola_chlorophyll_data[a_sola_chlorophyll_data['intertidal_zone']=="middle"]['ng_chlorophyll_per_ug_protein'],
        a_sola_chlorophyll_data[a_sola_chlorophyll_data['intertidal_zone']=="high"]['ng_chlorophyll_per_ug_protein']]

# using the posthoc_dunn() function
p_values= sp.posthoc_dunn(data, p_adjust = 'holm')

print(p_values)

### Intertidal Zone Box Plot
Summary box plot(s) that visualize variable mean and SEM grouped for each tidal zone and across time

In [ ]:
# algal density
intertidal_box_plot(a_sola_algae_data, 'num_cells_per_ug_protein', yaxis='Algal Cells/ug Animal Protein', title='Tidal Zone on Algal Density', save_path='plots/algae_intertidal_box_plot.png', box_color='#4eb3d3')

In [ ]:
# chlorophyll concentration
intertidal_box_plot(a_sola_chlorophyll_data, 'ng_chlorophyll_per_ug_protein', yaxis='ng Chl α/ug Animal Protein',title='Tidal Zone on Chlorophyll α', save_path='plots/chlorophyll_intertidal_box_plot.png', box_color='#2c7fb8')